# Corn Futures Data Builder

This notebook isolates the corn-price ingestion task from the main project notebook.

Goals:
- probe what Yahoo Finance actually provides for corn futures history
- download `ZC=F` in chunks when requested
- save a canonical local CSV cache for the project notebook
- keep the final-project notebook on a simple CSV-only path by default


In [ ]:
from pathlib import Path
from datetime import timedelta
import os
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/Users/jlaw/projects/stern/systematic-investing/.mplconfig")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path('/Users/jlaw/projects/stern/systematic-investing')
DATA_ROOT = PROJECT_ROOT / 'data' / 'ag_futures'
RAW_FUTURES_DIR = DATA_ROOT / 'raw' / 'futures'
PROCESSED_DIR = DATA_ROOT / 'processed'

RAW_FUTURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_START_DATE = '2000-02-18'
TARGET_END_DATE = pd.Timestamp.today().normalize().strftime('%Y-%m-%d')
CHUNK_YEARS = 3
REFRESH_CORN_FROM_YAHOO = False
RUN_CONTRACT_PROBE = False

RAW_CONTINUOUS_CACHE = RAW_FUTURES_DIR / 'zc_continuous_chunked_raw.csv'
PROCESSED_CONTINUOUS_CACHE = PROCESSED_DIR / 'corn_continuous_back_adjusted.csv'
CONTRACT_PROBE_CACHE = RAW_FUTURES_DIR / 'zc_contract_probe_results.csv'

ROLL_MONTHS = [3, 5, 7, 9, 12]
CONTRACT_SAMPLE_SYMBOLS = [
    'ZCH00.CBT', 'ZCK00.CBT', 'ZCN00.CBT', 'ZCU00.CBT', 'ZCZ00.CBT',
    'ZCH05.CBT', 'ZCH10.CBT', 'ZCH15.CBT', 'ZCH20.CBT', 'ZCH24.CBT',
]


In [ ]:
def normalize_yfinance_frame(frame):
    data = frame.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    keep = [col for col in ['Open', 'High', 'Low', 'Close', 'Volume'] if col in data.columns]
    return data[keep].dropna().copy()


def back_adjust_continuous_futures(frame):
    data = frame.copy()
    data['month'] = data.index.month
    data['is_roll_switch'] = data['month'].isin(ROLL_MONTHS) & (data['month'] != data['month'].shift(1))
    data['daily_change'] = data['Close'].diff()
    data['typical_change'] = (data['daily_change'].shift(1) + data['daily_change'].shift(-1)) / 2.0
    data['roll_gap'] = 0.0
    data.loc[data['is_roll_switch'], 'roll_gap'] = (
        data.loc[data['is_roll_switch'], 'daily_change'] - data.loc[data['is_roll_switch'], 'typical_change']
    )
    data['roll_gap'] = data['roll_gap'].fillna(0.0)
    data['cum_adjustment'] = data['roll_gap'].iloc[::-1].cumsum().iloc[::-1].shift(-1).fillna(0.0)
    data['Adj_Close_Futures'] = data['Close'] + data['cum_adjustment']
    data['futures_ret'] = data['Adj_Close_Futures'].pct_change()
    return data


def generate_chunk_windows(start_date, end_date, chunk_years=3):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    windows = []
    cursor = start
    while cursor <= end:
        window_end = min(cursor + pd.DateOffset(years=chunk_years), end + pd.Timedelta(days=1))
        windows.append((cursor.normalize(), pd.Timestamp(window_end).normalize()))
        cursor = window_end
    return windows


def download_continuous_corn_in_chunks(start_date, end_date, chunk_years=3):
    frames = []
    windows = generate_chunk_windows(start_date, end_date, chunk_years=chunk_years)
    print(f'Chunk windows: {len(windows)}')
    for idx, (window_start, window_end) in enumerate(windows, start=1):
        print(f'[{idx}/{len(windows)}] Downloading ZC=F from {window_start.date()} to {window_end.date()}')
        frame = yf.download(
            'ZC=F',
            start=window_start.strftime('%Y-%m-%d'),
            end=window_end.strftime('%Y-%m-%d'),
            auto_adjust=False,
            progress=False,
            actions=False,
        )
        frame = normalize_yfinance_frame(frame)
        if frame.empty:
            print('  -> empty chunk')
            continue
        print(f"  -> rows={len(frame)} first={frame.index.min().date()} last={frame.index.max().date()}")
        frames.append(frame)
    if not frames:
        raise RuntimeError('Yahoo returned no continuous corn data across all requested windows.')
    combined = pd.concat(frames).sort_index()
    combined = combined[~combined.index.duplicated(keep='last')]
    return combined


def probe_contract_samples(symbols):
    rows = []
    for symbol in symbols:
        try:
            frame = yf.download(symbol, period='3mo', auto_adjust=False, progress=False, actions=False)
            frame = normalize_yfinance_frame(frame)
            rows.append(
                {
                    'symbol': symbol,
                    'rows': int(len(frame)),
                    'first_date': frame.index.min().date().isoformat() if not frame.empty else None,
                    'last_date': frame.index.max().date().isoformat() if not frame.empty else None,
                }
            )
        except Exception as exc:
            rows.append({'symbol': symbol, 'rows': 0, 'first_date': None, 'last_date': None, 'error': str(exc)})
    return pd.DataFrame(rows)


In [ ]:
if RUN_CONTRACT_PROBE:
    probe_results = probe_contract_samples(CONTRACT_SAMPLE_SYMBOLS)
    probe_results.to_csv(CONTRACT_PROBE_CACHE, index=False)
    print(probe_results.to_string(index=False))
else:
    print('Contract probe skipped. Set RUN_CONTRACT_PROBE = True to test individual Yahoo symbols.')

if RAW_CONTINUOUS_CACHE.exists() and not REFRESH_CORN_FROM_YAHOO:
    corn_raw = pd.read_csv(RAW_CONTINUOUS_CACHE, parse_dates=['Date']).set_index('Date').sort_index()
    print(f'Loaded raw continuous corn cache from {RAW_CONTINUOUS_CACHE}')
else:
    corn_raw = download_continuous_corn_in_chunks(TARGET_START_DATE, TARGET_END_DATE, chunk_years=CHUNK_YEARS)
    corn_raw.to_csv(RAW_CONTINUOUS_CACHE, index_label='Date')
    print(f'Saved raw continuous corn cache to {RAW_CONTINUOUS_CACHE}')

corn = back_adjust_continuous_futures(corn_raw)
corn.to_csv(PROCESSED_CONTINUOUS_CACHE, index_label='Date')

print('Coverage summary')
print('rows', len(corn))
print('first_date', corn.index.min().date())
print('last_date', corn.index.max().date())
print(f'Saved processed corn cache to {PROCESSED_CONTINUOUS_CACHE}')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
corn[['Close', 'Adj_Close_Futures']].plot(ax=axes[0], linewidth=1.5)
axes[0].set_title('Corn continuous series: raw vs back-adjusted')
axes[0].grid(True, alpha=0.3)
roll_view = corn.loc[corn['is_roll_switch'], ['Close', 'Adj_Close_Futures', 'roll_gap']]
if not roll_view.empty:
    roll_view[['Close', 'Adj_Close_Futures']].plot(ax=axes[1], marker='o')
axes[1].set_title('Roll-switch checkpoints')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
